# B21 - Low-Rank Adaptation (LoRA)
$W_0 = \begin{bmatrix}1&2\\3&4\end{bmatrix}$ (frozen), $A = \begin{bmatrix}1&2\end{bmatrix}$ (r x d_in = 1x2), $B = \begin{bmatrix}1\\2\end{bmatrix}$ (d_out x r = 2x1), r = 1, $\alpha$ = 2

$\Delta W = \frac{\alpha}{r} BA = 2\begin{bmatrix}1&2\\2&4\end{bmatrix} = \begin{bmatrix}2&4\\4&8\end{bmatrix}$, so $W_{new} = W_0 + \Delta W = \begin{bmatrix}3&6\\7&12\end{bmatrix}$

$h = W_0x + \frac{\alpha}{r}BAx$

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

In [ ]:
# 1. LoRA architecture diagram
from matplotlib.patches import FancyBboxPatch

def box(ax, x, y, w, h, text, color):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02", fc=color, ec='black'))
    ax.text(x + w / 2, y + h / 2, text, ha='center', va='center', fontsize=10)

def arrow(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle='->', lw=1.5))

fig, ax = plt.subplots(figsize=(9, 6)); ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis('off')
box(ax, 4, 0.3, 2, 0.9, "Input x (d_in=2)", '#d9e8fb')
box(ax, 1, 4, 3, 1.4, "Pretrained W0 (2x2)\nFROZEN", '#cccccc')
box(ax, 6, 2.5, 3, 1, "A (r x d_in = 1x2)\ntrainable", '#c8f0c8')
box(ax, 6, 4.3, 3, 1, "B (d_out x r = 2x1)\ntrainable", '#c8f0c8')
box(ax, 6, 6.1, 3, 0.9, "Scale  alpha/r = 2", '#fde2b8')
box(ax, 4.4, 7.6, 1.2, 0.9, "+", '#ffffff')
box(ax, 4, 9, 2, 0.8, "Output h", '#d9e8fb')
arrow(ax, 5, 1.2, 2.5, 4); arrow(ax, 5, 1.2, 7.5, 2.5)
arrow(ax, 7.5, 3.5, 7.5, 4.3); arrow(ax, 7.5, 5.3, 7.5, 6.1)
arrow(ax, 2.5, 5.4, 4.6, 7.8); arrow(ax, 7.5, 7.0, 5.4, 7.8); arrow(ax, 5, 8.5, 5, 9)
ax.set_title("LoRA: h = W0 x + (alpha/r) B A x"); plt.show()

In [ ]:
W0 = tf.constant([[1., 2.], [3., 4.]])
A = tf.Variable([[1., 2.]])          # shape (r, d_in)  = (1, 2)
B = tf.Variable([[1.], [2.]])        # shape (d_out, r) = (2, 1)
r, alpha = 1, 2.0
d_out, d_in = W0.shape

# 2. Rank condition
print(f"r = {r}, min(d_out, d_in) = {min(d_out, d_in)} -> r < min(d_out, d_in)? {r < min(d_out, d_in)}")

In [ ]:
# 3. LoRA weight update  dW = (alpha / r) * B A
delta_W = (alpha / r) * tf.matmul(B, A)
print("Delta W =\n", delta_W.numpy())

# 4. Updated weight matrix
W_new = W0 + delta_W
print("W_new = W0 + Delta W =\n", W_new.numpy())

In [ ]:
# 5. Parameter counts
original_params = int(tf.size(W0))
lora_params = int(tf.size(A) + tf.size(B))
print("Original (frozen) parameters :", original_params)
print("Trainable LoRA parameters    :", lora_params, f"(A: {int(tf.size(A))}, B: {int(tf.size(B))})")
# For large layers LoRA saves a lot, e.g. d = 1024, r = 8:
d, rr = 1024, 8
print(f"Example d={d}, r={rr}: full = {d*d:,}  vs  LoRA = {2*d*rr:,}")

In [ ]:
# LoRA as a Keras layer: W0 frozen, A and B trainable
class LoRADense(layers.Layer):
    def __init__(self, W0, A, B, alpha, r):
        super().__init__()
        self.W0 = tf.Variable(W0, trainable=False, name='W0')
        self.A = tf.Variable(A, trainable=True, name='A')
        self.B = tf.Variable(B, trainable=True, name='B')
        self.scale = alpha / r

    def call(self, x):                     # x: (batch, d_in)
        frozen = tf.matmul(x, self.W0, transpose_b=True)
        lora = self.scale * tf.matmul(tf.matmul(x, self.A, transpose_b=True), self.B, transpose_b=True)
        return frozen + lora

layer = LoRADense(W0, A.numpy(), B.numpy(), alpha, r)
x = tf.constant([[1., 1.]])
print("Output with LoRA layer:", layer(x).numpy(), "| W_new @ x:", tf.matmul(x, W_new, transpose_b=True).numpy())
print("Trainable variables:", [v.name for v in layer.trainable_variables])
print("Non-trainable variables:", [v.name for v in layer.non_trainable_variables])